# **Coleta de dados**

In [1]:
#Bases de Dados 
from pathlib import Path
import pandas as pd
#import duckdb

# Visualização
import plotly.express as px

# Modelagem 
import numpy as np 
from typing import List, Optional

In [2]:
# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [3]:
path = Path("/home/carolina/Área de trabalho/vigimed/data/03_gold")
# Caminhos completos para o DuckDB (parâmetros nomeados não expandem expressões)
path_atc = str(path / 'dim_atc/dim_atc.parquet')
path_fat_medicamentos = str(path / 'fat_medicamentos/fat_medicamentos.parquet')
path_dim_notificacoes = str(path / 'dim_notificacoes/dim_notificacoes.parquet')
path_fat_reacoes = str(path / 'fat_reacoes/fat_reacoes.parquet')
path_dim_soc_llt = str(path / 'dim_soc_llt/dim_soc_llt.parquet')

In [4]:
%sql ROLLBACK;

Running query in 'duckdb:///:memory:'

,Success


In [5]:
%%sql
DROP TABLE IF EXISTS dim_atc;
CREATE TABLE dim_atc AS
SELECT * FROM read_parquet(:path_atc);

DROP TABLE IF EXISTS fat_medicamentos;
CREATE TABLE fat_medicamentos AS
SELECT *
FROM read_parquet(:path_fat_medicamentos);

DROP TABLE IF EXISTS dim_notificacoes;
CREATE TABLE dim_notificacoes AS
SELECT *
FROM read_parquet(:path_dim_notificacoes);

DROP TABLE IF EXISTS fat_reacoes;
CREATE TABLE fat_reacoes AS
SELECT *
FROM read_parquet(:path_fat_reacoes);

DROP TABLE IF EXISTS dim_soc_llt;
CREATE TABLE dim_soc_llt AS
SELECT *
FROM read_parquet(:path_dim_soc_llt);
 

Running query in 'duckdb:///:memory:'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


# **5️⃣ Entendimento dos dados**

### dim_notificacoes

In [ ]:
%%sql 
select * from dim_notificacoes limit 2

In [ ]:
%%sql 
select IDENTIFICACAO_NOTIFICACAO,count(1) as qt
from dim_notificacoes 
group by 1
having qt > 1
limit 10

In [ ]:
%%sql 
select * from dim_notificacoes 
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

### fat_medicamentos

In [ ]:
%%sql 
select * from fat_medicamentos 
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

### fat_reacoes

In [ ]:
%%sql 
select * from fat_reacoes limit 2

In [ ]:
%%sql 
select * from fat_reacoes 
left join dim_soc_llt using (LLT_CHAVE)
where IDENTIFICACAO_NOTIFICACAO = 'BR-ANVISA-300171130'

### dim_soc_llt

In [ ]:
%%sql 
select * from dim_soc_llt limit 2

### 6.1 Aplicacao de regras de negócio

In [ ]:
%%sql
DROP TABLE IF EXISTS fat_analytics;

CREATE TABLE fat_analytics AS
WITH fat_med AS (
    SELECT 
        m.IDENTIFICACAO_NOTIFICACAO, 
        atc.ATC_CODE_LEVEL_4_LEVEL_NAME,
        CASE 
            WHEN atc.ATC_CODE_LEVEL_4 IN ('L04AB', 'L04AA', 'L01FA', 'L04AC', 'L04AF') 
                THEN (
                    CASE 
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%rituximab%' THEN 'L01FA01_Rituximabe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%abatacept%' THEN 'L04AA24_Abatacepte'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%etanercept%' THEN 'L04AB01_Etanercepte'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%infliximab%' THEN 'L04AB02_Infliximabe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%adalimumab%' THEN 'L04AB04_Adalimumabe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%certolizumab%' THEN 'L04AB05_Certolizumabe Pegol'   
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%golimumab%' THEN 'L04AB06_Golimumabe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tocilizumab%' THEN 'L04AC07_Tocilizumabe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%tofacitinib%' THEN 'L04AF01_Tofacitinibe'
                        WHEN LOWER(m.PRINCIPIOS_ATIVOS_WHODRUG) LIKE '%baricitinib%' THEN 'L04AF02_Baricitinibe'
                        ELSE 'Outros'
                    END
                )
            ELSE 'Outros'
        END AS ATC_LEVEL_5  
    FROM fat_medicamentos m
    INNER JOIN dim_atc AS atc
        ON m.ATC_CODE_LEVEL_4 = atc.ATC_CODE_LEVEL_4 AND atc.ATC_LEVEL = 4
),
fat AS (
    SELECT 
        strftime(TRY_CAST(dim.DATA_INCLUSAO_SISTEMA AS DATE), '%Y-%m') AS DATA_NOTIFICACAO,
        dim.IDENTIFICACAO_NOTIFICACAO AS ID,
        dim.UF_VALOR AS UF,
        dim.SEXO_VALOR AS SEXO,
        fat_med.ATC_LEVEL_5,
        ds.LLT_NAME AS REACAO_LLT,
        ds.PT_NAME AS REACAO_PT
    FROM dim_notificacoes AS dim
    INNER JOIN fat_med ON dim.IDENTIFICACAO_NOTIFICACAO = fat_med.IDENTIFICACAO_NOTIFICACAO
    INNER JOIN fat_reacoes fr ON dim.IDENTIFICACAO_NOTIFICACAO = fr.IDENTIFICACAO_NOTIFICACAO
    LEFT JOIN dim_soc_llt ds ON 
        '100' || LPAD(CAST(fr.LLT_CHAVE AS VARCHAR), 5, '0') = CAST(ds.LLT_CODE AS VARCHAR)
)
SELECT * FROM fat WHERE ATC_LEVEL_5 <> 'Outros'
UNION ALL 
SELECT * FROM fat AS a WHERE a.ATC_LEVEL_5 = 'Outros'
AND a.ID NOT IN (SELECT b.ID FROM fat AS b WHERE b.ATC_LEVEL_5 <> 'Outros');

### 6.2 Checagens

In [ ]:
%%sql
select * from fat_analytics limit 5

In [ ]:
%%sql
select ATC_LEVEL_5,count(*) as qt,count(distinct ID) as qt_notificacoes from fat_analytics group by ATC_LEVEL_5 

In [ ]:
%%sql
select id,count(1) from fat_analytics group by 1 having count(1) > 1 limit 2

In [ ]:
%%sql 
select * from fat_analytics 
where id ='BR-ANVISA-300315256'

### 6.3 Persistindo a Base

In [ ]:
%%sql df <<
select * from fat_analytics

In [ ]:
df.to_parquet(path/'fat_analytical/fat_analytical.parquet', index=False)
